In [0]:
from pyspark.sql import functions as fs
from delta.tables import DeltaTable


In [0]:
%run /Workspace/Users/juanesteban231espana@gmail.com/Pro_MusEntre/Init_Child_Schema/utilities

In [0]:
dbutils.widgets.text("catalog", "fmgc", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data customers")

data_source=dbutils.widgets.get("data_source")
catalog=dbutils.widgets.get("catalog")

base_path = f's3://childs-port-datalake/{data_source}/*.csv'
print(base_path)


In [0]:
df = (
    spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(base_path)
        .select("*", "_metadata.file_name", "_metadata.file_size")
        .withColumn("read_timestamp", fs.current_timestamp())
)

In [0]:
display(df.limit(10))

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")